# Sensitivity Analysis: Epsilon & Alpha with Outlier Diagnostics

Test different combinations of `epsilon` (Huber threshold) and `alpha` (Ridge strength) to understand:
- Impact on fit quality (RMSE, R²)
- Outlier detection sensitivity
- Visual inspection of outliers in U-T space for each interval

**Focus: Visual outlier diagnostics for each parameter combination**

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import linregress
from IPython.display import display

# Project imports
from degradation_toolbox.Urc.Urc1_huber import Urc1BaseHuber
from degradation_toolbox.Urc.Urc1 import Urc1
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess
from master_arbeit_Di.utils.stability_evaluation import evaluate_urc_stability, compare_urc_models

In [2]:
# -- Load & preprocess data --
# Keep this cell runnable even when executed before the config cell.
DATASET_PATH = Path(r"..\..\explore_data\G1M1_new.parquet")
PREPROCESS_OUTPUT_DIR = Path(r"..\..\explore_data\output")
OUTPUT_DIR = Path(r"..\plots\huber")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REF_CONFIGS = {
    "Low": {
        "Iref": 0.3,
        "Tref": 58,
        "OHref": 11,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 100,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.48,
        "Tref": 57,
        "OHref": 100,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv",
    },
}

preprocessor = GMpreprocess(file_path=str(DATASET_PATH), output_dir=str(PREPROCESS_OUTPUT_DIR))
data = preprocessor.run()
extracted_name = preprocessor.name

# Verify data loaded successfully
if data is None or data.empty:
    raise ValueError("Data loading failed! Check that the parquet file exists and is valid.")

print(f"Dataset path: {DATASET_PATH.resolve()}")
print(f"GT refs: {list(REF_CONFIGS.keys())}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"✓ Dataset: {extracted_name}")
print(f"✓ Data shape: {data.shape}")
print(f"✓ Columns: {list(data.columns)}")

=== 1. Loading & Preprocessing: G1M1_new ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_1' -> ID: '1'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\..\explore_data\output\G1M1_new_20260501_145712.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset path: C:\Users\Z0057NPT\Documents\MA_code\explore_data\G1M1_new.parquet
GT refs: ['Low', 'Medium', 'High']
Output dir: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\plots\huber
✓ Dataset: G1M1_new
✓ Data shape: (2529217, 3)
✓ Columns: ['currentDensity', 'temperature', 'voltage']


In [3]:
# -- Unified-compare-aligned configuration (G1M1) --
DATASET_PATH = Path(r"..\..\explore_data\G1M1_new.parquet")
PREPROCESS_OUTPUT_DIR = Path(r"..\..\explore_data\output")
OUTPUT_DIR = Path(r"..\plots\huber")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REF_CONFIGS = {
    "Low": {
        "Iref": 0.3,
        "Tref": 58,
        "OHref": 11,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 100,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.48,
        "Tref": 57,
        "OHref": 100,
        "gt_file": r"..\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv",
    },
}

common_config = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
    "data_filter_h_since_last_start_min": 0.5,
}

SHOW_GT_METRICS = True

In [4]:
# ── Preprocess once and reuse in all model fits (important for fair sensitivity comparison) ──
shared = Urc1.preprocess_once(
    data,
    i_off=common_config["i_off"],
    u_off=common_config["u_off"],
    data_filter_i_min=common_config["data_filter_i_min"],
    data_filter_U_min=common_config["data_filter_U_min"],
    data_filter_U_max=common_config["data_filter_U_max"],
    data_filter_T_min=common_config["data_filter_T_min"],
    data_filter_T_max=common_config["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared)}")

Shared preprocessed rows: 1069062


## Parameter Grid Setup

Define ranges for sensitivity analysis:
- **Epsilon**: Huber threshold (smaller = more outlier-sensitive)
- **Alpha**: Ridge regularization strength (larger = stronger L2 penalty)

In [5]:
# ── Parameter grid ──
epsilon_values = [1.0, 1.35, 2.0, 10**6] # Huber threshold
alpha_values = [0.0, 0.01, 0.1, 1, 10, 100]  # Ridge strength

print(f"Total combinations: {len(epsilon_values)} × {len(alpha_values)} = {len(epsilon_values) * len(alpha_values)}")
print(f"Epsilon range: {epsilon_values}")
print(f"Alpha range: {alpha_values}")

Total combinations: 4 × 6 = 24
Epsilon range: [1.0, 1.35, 2.0, 1000000]
Alpha range: [0.0, 0.01, 0.1, 1, 10, 100]


<!-- ## Run All Parameter Combinations

Fit models for all (epsilon, alpha) pairs and store results. -->

In [6]:
# ── Run all combinations ──
results = {}  # Key: (epsilon, alpha), Value: Urc1BaseHuber instance

for eps in epsilon_values:
    for alpha in alpha_values:
        label = f"ε={eps}, α={alpha}"
        print(f"\n{'='*60}\n  {label}\n{'='*60}")
        
        urc = Urc1BaseHuber(
            data=data,
            name=f"{extracted_name}_{label}",
            epsilon=eps,
            alpha=alpha,
            preprocessed_data=shared,
            **common_config
)
        results[(eps, alpha)] = urc

print(f"\n✓ Completed {len(results)} model fits")


  ε=1.0, α=0.0
[Huber eps=1.0 alpha=0.0] Fitting Stats: 474 intervals low data, 0 fit failed.
  [Huber] epsilon=1.0, alpha=0.0

  ε=1.0, α=0.01
[Huber+Ridge eps=1.0 alpha=0.01] Fitting Stats: 474 intervals low data, 0 fit failed.
  [Huber+Ridge] epsilon=1.0, alpha=0.01

  ε=1.0, α=0.1
[Huber+Ridge eps=1.0 alpha=0.1] Fitting Stats: 474 intervals low data, 0 fit failed.
  [Huber+Ridge] epsilon=1.0, alpha=0.1

  ε=1.0, α=1
[Huber+Ridge eps=1.0 alpha=1] Fitting Stats: 474 intervals low data, 0 fit failed.
  [Huber+Ridge] epsilon=1.0, alpha=1

  ε=1.0, α=10
[Huber+Ridge eps=1.0 alpha=10] Fitting Stats: 474 intervals low data, 0 fit failed.
  [Huber+Ridge] epsilon=1.0, alpha=10

  ε=1.0, α=100
[Huber+Ridge eps=1.0 alpha=100] Fitting Stats: 474 intervals low data, 1 fit failed.
  [Huber+Ridge] epsilon=1.0, alpha=100

  ε=1.35, α=0.0
[Huber eps=1.35 alpha=0.0] Fitting Stats: 474 intervals low data, 0 fit failed.
  [Huber] epsilon=1.35, alpha=0.0

  ε=1.35, α=0.01
[Huber+Ridge eps=1.35 alpha=0

In [7]:
# ── Run Baseline (Urc1 OLS) ──
# Make this cell runnable even if previous import cell was not executed.
from degradation_toolbox.Urc.Urc1 import Urc1

print("\n" + "="*60)
print("  Running Baseline Model (Urc1 OLS)")
print("="*60)

urc_baseline = Urc1(
    data=data,
    name=f"{extracted_name}_Baseline",
    preprocessed_data=shared,
    **common_config
)

print(f"✓ Baseline model completed")
print(f"  Reliable intervals: {len(urc_baseline.fitting_results_reliable)}")

# Get baseline metrics for each Iref
baseline_metrics = {}
for i_ref in common_config["Iref"]:
    models_dict = {"Baseline (Urc1 OLS)": urc_baseline}
    metrics_df = compare_urc_models(models_dict, i_target=i_ref)
    if not metrics_df.empty:
        baseline_metrics[i_ref] = {
            "RMSE (mV)": metrics_df.loc[metrics_df.index[0], "RMSE (mV)"],
            "Outlier (%)": metrics_df.loc[metrics_df.index[0], "Outlier (%)"],
        }
        print(f"\nBaseline @ Iref={i_ref}:")
        print(f"  RMSE: {baseline_metrics[i_ref]['RMSE (mV)']:.2f} mV")
        print(f"  Outlier %: {baseline_metrics[i_ref]['Outlier (%)']:.2f}%")


  Running Baseline Model (Urc1 OLS)
✓ Baseline model completed
  Reliable intervals: 1098

Baseline @ Iref=0.3:
  RMSE: 18.89 mV
  Outlier %: 0.55%

Baseline @ Iref=1.0:
  RMSE: 18.87 mV
  Outlier %: 0.91%

Baseline @ Iref=1.48:
  RMSE: 55.14 mV
  Outlier %: 1.46%


## Run Baseline Model (Urc1 OLS)

Fit the baseline Urc1 model (OLS without robustness) for comparison.

## 1. Summary Heatmap: RMSE vs (Epsilon, Alpha)

Visualize median RMSE across all parameter combinations.

In [8]:
# ── Build summary matrices per Iref using compare_urc_models ──
iref_list = common_config["Iref"]
compare_df_all = []  # collect comparison tables for all Iref
rmse_matrices = {}
outlier_matrices = {}

for i_ref in iref_list:
    rmse_matrix = np.zeros((len(epsilon_values), len(alpha_values)))
    outlier_matrix = np.zeros((len(epsilon_values), len(alpha_values)))

    for i, eps in enumerate(epsilon_values):
        for j, alpha in enumerate(alpha_values):
            models_dict = {f"ε={eps}, α={alpha}": results[(eps, alpha)]}
            metrics_df = compare_urc_models(models_dict, i_target=i_ref)

            if not metrics_df.empty:
                rmse_matrix[i, j] = metrics_df.loc[metrics_df.index[0], "RMSE (mV)"]
                outlier_matrix[i, j] = metrics_df.loc[metrics_df.index[0], "Outlier (%)"]
            else:
                rmse_matrix[i, j] = np.nan
                outlier_matrix[i, j] = np.nan

    rmse_matrices[i_ref] = rmse_matrix.copy()
    outlier_matrices[i_ref] = outlier_matrix.copy()

    # Get baseline value for reference
    baseline_rmse = baseline_metrics[i_ref]["RMSE (mV)"]
    baseline_outlier = baseline_metrics[i_ref]["Outlier (%)"]

    # Plot RMSE heatmap with baseline reference
    fig = go.Figure(data=go.Heatmap(
        z=rmse_matrix,
        x=[f"α={a}" for a in alpha_values],
        y=[f"ε={e}" for e in epsilon_values],
        colorscale="Viridis",
        text=np.round(rmse_matrix, 4),
        texttemplate="%{text}",
        textfont={"size": 10},
        colorbar=dict(title=f"RMSE (mV)"),
    ))

    fig.update_layout(
        title=f"{extracted_name} — RMSE Heatmap @ Iref={i_ref}<br><sub>Baseline (Urc1 OLS): {baseline_rmse:.2f} mV</sub>",
        xaxis_title="Alpha (Ridge Strength)",
        yaxis_title="Epsilon (Huber Threshold)",
        height=450,
        template="plotly_white",
    )
    fig.show()

    # RMSE delta vs baseline (explicit baseline comparison)
    rmse_delta = rmse_matrix - baseline_rmse
    fig = go.Figure(data=go.Heatmap(
        z=rmse_delta,
        x=[f"α={a}" for a in alpha_values],
        y=[f"ε={e}" for e in epsilon_values],
        colorscale="RdBu",
        zmid=0,
        text=np.round(rmse_delta, 3),
        texttemplate="%{text}",
        textfont={"size": 10},
        colorbar=dict(title="ΔRMSE vs Baseline (mV)"),
    ))

    fig.update_layout(
        title=f"{extracted_name} — ΔRMSE vs Baseline @ Iref={i_ref}<br><sub>Negative is better than baseline</sub>",
        xaxis_title="Alpha (Ridge Strength)",
        yaxis_title="Epsilon (Huber Threshold)",
        height=450,
        template="plotly_white",
    )
    fig.show()

    # Plot Outlier % heatmap with baseline reference
    fig = go.Figure(data=go.Heatmap(
        z=outlier_matrix,
        x=[f"α={a}" for a in alpha_values],
        y=[f"ε={e}" for e in epsilon_values],
        colorscale="Reds",
        text=np.round(outlier_matrix, 1),
        texttemplate="%{text}%",
        textfont={"size": 10},
        colorbar=dict(title=f"Outlier (%)"),
    ))

    fig.update_layout(
        title=f"{extracted_name} — Outlier % Heatmap @ Iref={i_ref}<br><sub>Baseline (Urc1 OLS): {baseline_outlier:.2f}%</sub>",
        xaxis_title="Alpha (Ridge Strength)",
        yaxis_title="Epsilon (Huber Threshold)",
        height=450,
        template="plotly_white",
    )
    fig.show()

    # Outlier delta vs baseline (explicit baseline comparison)
    outlier_delta = outlier_matrix - baseline_outlier
    fig = go.Figure(data=go.Heatmap(
        z=outlier_delta,
        x=[f"α={a}" for a in alpha_values],
        y=[f"ε={e}" for e in epsilon_values],
        colorscale="RdBu",
        zmid=0,
        text=np.round(outlier_delta, 2),
        texttemplate="%{text}",
        textfont={"size": 10},
        colorbar=dict(title="ΔOutlier vs Baseline (%)"),
    ))

    fig.update_layout(
        title=f"{extracted_name} — ΔOutlier % vs Baseline @ Iref={i_ref}<br><sub>Negative is better than baseline</sub>",
        xaxis_title="Alpha (Ridge Strength)",
        yaxis_title="Epsilon (Huber Threshold)",
        height=450,
        template="plotly_white",
    )
    fig.show()

    # ---- Full metrics comparison: Baseline vs Best Huber+Ridge ----
    min_idx = np.unravel_index(np.nanargmin(rmse_matrix), rmse_matrix.shape)
    best_eps = epsilon_values[min_idx[0]]
    best_alpha = alpha_values[min_idx[1]]
    best_model = results[(best_eps, best_alpha)]

    compare_df = compare_urc_models(
        {
            "Baseline (Urc1 OLS)": urc_baseline,
            f"Best Huber+Ridge (ε={best_eps}, α={best_alpha})": best_model,
        },
        i_target=i_ref,
    )

    compare_df = compare_df.copy()
    compare_df.insert(0, "Iref", i_ref)
    compare_df_all.append(compare_df)

    print(f"\n{'='*60}")
    print(f"  Iref = {i_ref} A/cm² — Full Metrics Comparison")
    print(f"{'='*60}")
    display(compare_df)

    # Also print quick deltas for RMSE / Outlier
    best_rmse = compare_df.loc[compare_df["Model Name"].str.contains(r"Best Huber\+Ridge"), "RMSE (mV)"].values[0]
    best_outlier = compare_df.loc[compare_df["Model Name"].str.contains(r"Best Huber\+Ridge"), "Outlier (%)"].values[0]

    print(f"\nBaseline (Urc1 OLS):")
    print(f"  RMSE: {baseline_rmse:.2f} mV")
    print(f"  Outlier %: {baseline_outlier:.2f}%")
    print(f"\nBest Huber+Ridge (min RMSE):")
    print(f"  ε={best_eps}, α={best_alpha}")
    print(f"  RMSE: {best_rmse:.2f} mV (Δ={best_rmse-baseline_rmse:+.2f} mV)")
    print(f"  Outlier %: {best_outlier:.2f}% (Δ={best_outlier-baseline_outlier:+.2f}%)")

# ---- Combined comparison table across all Iref ----
compare_df_summary = pd.concat(compare_df_all, ignore_index=True)
print("\n" + "="*60)
print("  Combined Comparison Table (All Iref)")
print("="*60)
display(compare_df_summary)


  Iref = 0.3 A/cm² — Full Metrics Comparison


,Iref,Model Name,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),Cond Median log10,Cond P95 log10,Cond Mean,Cond Min,Cond Max,Cond Count
0,0.3,Baseline (Urc1 OLS),0.3,1098,18.885,0.048,0.925,6,0.55,390.861,4.105,4.687,5.255,7.161180e+04,9463.6,8.665571e+05,1098
1,0.3,"Best Huber+Ridge (ε=1.35, α=0.0)",0.3,1085,17.749,0.045,0.930,6,0.55,369.306,2.584,4.955,13.282,5.908058e+12,9580.3,1.696310e+15,1286



Baseline (Urc1 OLS):
  RMSE: 18.89 mV
  Outlier %: 0.55%

Best Huber+Ridge (min RMSE):
  ε=1.35, α=0.0
  RMSE: 17.75 mV (Δ=-1.14 mV)
  Outlier %: 0.55% (Δ=+0.00%)



  Iref = 1.0 A/cm² — Full Metrics Comparison


,Iref,Model Name,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),Cond Median log10,Cond P95 log10,Cond Mean,Cond Min,Cond Max,Cond Count
0,1.0,Baseline (Urc1 OLS),1.0,1098,18.873,0.048,0.959,10,0.91,141.761,4.137,4.687,5.255,7.161180e+04,9463.6,8.665571e+05,1098
1,1.0,"Best Huber+Ridge (ε=2.0, α=0.0)",1.0,1087,18.851,0.048,0.959,9,0.83,141.508,2.716,4.895,13.286,5.867878e+12,9403.2,1.682859e+15,1286



Baseline (Urc1 OLS):
  RMSE: 18.87 mV
  Outlier %: 0.91%

Best Huber+Ridge (min RMSE):
  ε=2.0, α=0.0
  RMSE: 18.85 mV (Δ=-0.02 mV)
  Outlier %: 0.83% (Δ=-0.08%)



  Iref = 1.48 A/cm² — Full Metrics Comparison


,Iref,Model Name,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),Cond Median log10,Cond P95 log10,Cond Mean,Cond Min,Cond Max,Cond Count
0,1.48,Baseline (Urc1 OLS),1.48,1098,55.143,0.14,0.914,16,1.46,326.556,5.353,4.687,5.255,7.161180e+04,9463.6,8.665571e+05,1098
1,1.48,"Best Huber+Ridge (ε=1000000, α=0.0)",1.48,1098,55.143,0.14,0.914,16,1.46,326.556,3.621,4.745,13.288,5.853256e+12,9463.6,1.686499e+15,1286



Baseline (Urc1 OLS):
  RMSE: 55.14 mV
  Outlier %: 1.46%

Best Huber+Ridge (min RMSE):
  ε=1000000, α=0.0
  RMSE: 55.14 mV (Δ=+0.00 mV)
  Outlier %: 1.46% (Δ=+0.00%)

  Combined Comparison Table (All Iref)


,Iref,Model Name,Target Current (A/cm2),Data Points (n),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),Cond Median log10,Cond P95 log10,Cond Mean,Cond Min,Cond Max,Cond Count
0,0.30,Baseline (Urc1 OLS),0.30,1098,18.885,0.048,0.925,6,0.55,390.861,4.105,4.687,5.255,7.161180e+04,9463.6,8.665571e+05,1098
1,0.30,"Best Huber+Ridge (ε=1.35, α=0.0)",0.30,1085,17.749,0.045,0.930,6,0.55,369.306,2.584,4.955,13.282,5.908058e+12,9580.3,1.696310e+15,1286
2,1.00,Baseline (Urc1 OLS),1.00,1098,18.873,0.048,0.959,10,0.91,141.761,4.137,4.687,5.255,7.161180e+04,9463.6,8.665571e+05,1098
3,1.00,"Best Huber+Ridge (ε=2.0, α=0.0)",1.00,1087,18.851,0.048,0.959,9,0.83,141.508,2.716,4.895,13.286,5.867878e+12,9403.2,1.682859e+15,1286
4,1.48,Baseline (Urc1 OLS),1.48,1098,55.143,0.140,0.914,16,1.46,326.556,5.353,4.687,5.255,7.161180e+04,9463.6,8.665571e+05,1098
5,1.48,"Best Huber+Ridge (ε=1000000, α=0.0)",1.48,1098,55.143,0.140,0.914,16,1.46,326.556,3.621,4.745,13.288,5.853256e+12,9463.6,1.686499e+15,1286


## 2. Outlier Percentage Heatmap

Average outlier % across all intervals for each parameter combination.

## 3. Detailed Comparison Table

All key metrics for each parameter combination.

In [9]:
# ── Build comparison table ──
rows = []
for eps in epsilon_values:
    for alpha in alpha_values:
        rel = results[(eps, alpha)].fitting_results_reliable
        total = len(results[(eps, alpha)].fitting_results.dropna(subset=["c1"]))
        n_rel = len(rel)

        row = {
            "Model": "Huber+Ridge",
            "Epsilon": eps,
            "Alpha": alpha,
            "Reliable": n_rel,
            "Total": total,
            "Reliable %": round(n_rel / max(total, 1) * 100, 1),
        }

        if not rel.empty:
            row["Median RMSE [mV]"] = round(rel["RMSE"].median() * 1000, 2)
            row["Median R²"] = round(rel["R2"].median(), 4)
            if "outlier_pct" in rel.columns:
                row["Mean Outlier %"] = round(rel["outlier_pct"].mean(), 2)
                row["Max Outlier %"] = round(rel["outlier_pct"].max(), 2)

        rows.append(row)

df_comparison = pd.DataFrame(rows)

# Baseline summary table (explicitly included in comparison tables)
baseline_total = len(urc_baseline.fitting_results.dropna(subset=["c1"]))
baseline_reliable = len(urc_baseline.fitting_results_reliable)
baseline_rows = []
for i_ref in common_config["Iref"]:
    baseline_rows.append(
        {
            "Model": "Baseline (Urc1 OLS)",
            "Iref": i_ref,
            "RMSE (mV)": round(baseline_metrics[i_ref]["RMSE (mV)"], 2),
            "Outlier (%)": round(baseline_metrics[i_ref]["Outlier (%)"], 2),
            "Reliable": baseline_reliable,
            "Total": baseline_total,
            "Reliable %": round(baseline_reliable / max(baseline_total, 1) * 100, 1),
        }
    )

df_baseline = pd.DataFrame(baseline_rows)

# Best Huber+Ridge vs baseline per Iref
best_vs_baseline_rows = []
for i_ref in common_config["Iref"]:
    rmse_matrix = rmse_matrices[i_ref]
    outlier_matrix = outlier_matrices[i_ref]

    min_idx = np.unravel_index(np.nanargmin(rmse_matrix), rmse_matrix.shape)
    best_eps = epsilon_values[min_idx[0]]
    best_alpha = alpha_values[min_idx[1]]
    best_rmse = rmse_matrix[min_idx]
    best_outlier = outlier_matrix[min_idx]

    baseline_rmse = baseline_metrics[i_ref]["RMSE (mV)"]
    baseline_outlier = baseline_metrics[i_ref]["Outlier (%)"]

    best_vs_baseline_rows.append(
        {
            "Iref": i_ref,
            "Baseline RMSE (mV)": round(baseline_rmse, 2),
            "Best ε": best_eps,
            "Best α": best_alpha,
            "Best RMSE (mV)": round(best_rmse, 2),
            "ΔRMSE vs Baseline (mV)": round(best_rmse - baseline_rmse, 2),
            "Baseline Outlier (%)": round(baseline_outlier, 2),
            "Best Outlier (%)": round(best_outlier, 2),
            "ΔOutlier vs Baseline (%)": round(best_outlier - baseline_outlier, 2),
        }
    )

df_best_vs_baseline = pd.DataFrame(best_vs_baseline_rows)

print("\n" + "=" * 60)
print("  Baseline Summary (Urc1 OLS)")
print("=" * 60)
display(df_baseline)

print("\n" + "=" * 60)
print("  Best Huber+Ridge vs Baseline (per Iref)")
print("=" * 60)
display(df_best_vs_baseline)

print("\n" + "=" * 60)
print("  Full Huber+Ridge Grid Summary")
print("=" * 60)
display(df_comparison)

# ── Stability evaluation for key parameter combinations ──
print("\n" + "="*60)
print("  Stability Comparison (with Baseline reference)")
print("="*60)

# Select a few representative combinations for detailed comparison
key_combinations = [
    (1.35, 0.0, "Huber only"),
    (1.35, 0.1, "Huber+Ridge"),
    (1.0, 0.0, "Conservative Huber"),
    (2.0, 0.1, "Relaxed Huber+Ridge"),
]

for eps, alpha, label in key_combinations:
    if (eps, alpha) in results:
        print(f"\n{'─'*60}")
        print(f"  {label}: ε={eps}, α={alpha}")
        print(f"{'─'*60}")

        # Evaluate at each Iref
        for i_ref in [1.0]:  # Example: show detailed metrics for Iref=1.0
            metrics = evaluate_urc_stability(results[(eps, alpha)], i_ref)
            if metrics:
                print(f"\n  @ Iref = {i_ref} A/cm²:")
                for k, v in metrics.items():
                    if k != "Target Current (A/cm2)":
                        print(f"    {k:<30} = {v}")

                print("\n  Baseline reference:")
                print(f"    RMSE (mV)                     = {baseline_metrics[i_ref]['RMSE (mV)']:.2f}")
                print(f"    Outlier (%)                   = {baseline_metrics[i_ref]['Outlier (%)']:.2f}")
        break  # Just show one for brevity; user can expand as needed


  Baseline Summary (Urc1 OLS)


,Model,Iref,RMSE (mV),Outlier (%),Reliable,Total,Reliable %
0,Baseline (Urc1 OLS),0.30,18.89,0.55,1098,1286,85.4
1,Baseline (Urc1 OLS),1.00,18.87,0.91,1098,1286,85.4
2,Baseline (Urc1 OLS),1.48,55.14,1.46,1098,1286,85.4



  Best Huber+Ridge vs Baseline (per Iref)


,Iref,Baseline RMSE (mV),Best ε,Best α,Best RMSE (mV),ΔRMSE vs Baseline (mV),Baseline Outlier (%),Best Outlier (%),ΔOutlier vs Baseline (%)
0,0.30,18.89,1.35,0.0,17.75,-1.14,0.55,0.55,0.00
1,1.00,18.87,2.00,0.0,18.85,-0.02,0.91,0.83,-0.08
2,1.48,55.14,1000000.00,0.0,55.14,0.00,1.46,1.46,0.00



  Full Huber+Ridge Grid Summary


,Model,Epsilon,Alpha,Reliable,Total,Reliable %,Median RMSE [mV],Median R²,Mean Outlier %,Max Outlier %
0,Huber+Ridge,1.00,0.00,1081,1286,84.1,3.97,0.9978,34.34,45.07
1,Huber+Ridge,1.00,0.01,1256,1286,97.7,3.69,0.9976,34.06,45.30
2,Huber+Ridge,1.00,0.10,1277,1286,99.3,3.71,0.9975,33.98,44.92
3,Huber+Ridge,1.00,1.00,1274,1286,99.1,4.08,0.9972,33.94,43.87
4,Huber+Ridge,1.00,10.00,1274,1286,99.1,4.94,0.9957,34.11,43.87
5,Huber+Ridge,1.00,100.00,1275,1285,99.2,5.56,0.9951,34.28,44.50
6,Huber+Ridge,1.35,0.00,1085,1286,84.4,3.94,0.9978,23.58,41.31
7,Huber+Ridge,1.35,0.01,1252,1286,97.4,3.68,0.9976,23.01,40.78
8,Huber+Ridge,1.35,0.10,1276,1286,99.2,3.71,0.9975,22.79,38.90
9,Huber+Ridge,1.35,1.00,1273,1286,99.0,4.09,0.9971,22.81,36.95



  Stability Comparison (with Baseline reference)

────────────────────────────────────────────────────────────
  Huber only: ε=1.35, α=0.0
────────────────────────────────────────────────────────────

  @ Iref = 1.0 A/cm²:
    Data Points (n)                = 1085
    RMSE (mV)                      = 18.978
    Slope Sigma (uV/h)             = 0.048
    Mono (Rank) [0-1]              = 0.958
    Outlier Count                  = 9
    Outlier (%)                    = 0.83
    Max Residual (mV)              = 146.775
    Mean SE (mV)                   = 2.629

  Baseline reference:
    RMSE (mV)                     = 18.87
    Outlier (%)                   = 0.91


## 4. Visual Outlier Inspection: U vs Fitted U

For a selected parameter combination, visualize outliers in actual vs fitted voltage space for each interval.

**Red points = outliers flagged by Huber loss**

In [10]:
# ── Select parameter combination to inspect ──
eps_inspect = 1.35
alpha_inspect = 0.01

urc_inspect = results[(eps_inspect, alpha_inspect)]
print(f"Inspecting: ε={eps_inspect}, α={alpha_inspect}")
print(f"Model: {urc_inspect.name}")

Inspecting: ε=1.35, α=0.01
Model: G1M1_new_ε=1.35, α=0.01


In [11]:
# ── Extract interval data with outlier flags ──
# Get a few representative intervals
rel = urc_inspect.fitting_results_reliable
intervals_to_plot = rel.index[:5].tolist()  # First 5 reliable intervals

print(f"Plotting {len(intervals_to_plot)} intervals: {intervals_to_plot}")

Plotting 5 intervals: [0, 1, 2, 3, 4]


In [12]:
# ── Plot U vs U_fitted with outlier highlighting ──
# Robust interval slicing without relying on `urc_inspect.intervals`

fig = make_subplots(
    rows=len(intervals_to_plot), cols=1,
    subplot_titles=[f"Interval {idx} (calh={rel.loc[idx, 'calh']:.0f}h)" 
                    for idx in intervals_to_plot],
    vertical_spacing=0.08,
)

for i, idx in enumerate(intervals_to_plot):
    # Get interval metadata from reliable results
    row_meta = rel.loc[idx]
    if isinstance(row_meta, pd.DataFrame):
        row_meta = row_meta.iloc[0]

    day_since_install = row_meta.get("day_since_install", np.nan)

    if pd.isna(day_since_install):
        day_since_install = row_meta["calh"] / 24.0

    current_day = urc_inspect.installation_time + pd.Timedelta(days=float(day_since_install))
    interval_end = current_day + pd.Timedelta(days=urc_inspect.len_interval)

    # Slice original preprocessed data in the same time window as model fitting
    interval_data = urc_inspect.data.query("@current_day <= index <= @interval_end")

    if interval_data.empty:
        continue

    # Get coefficients
    c1, c2, c3, c4, c5 = [row_meta[f"c{j}"] for j in range(1, 6)]

    # Build features exactly like training
    X, _ = urc_inspect._prepare_matrices(interval_data)

    # Compute fitted voltage in physical unit [V]
    U_ocv = -8.2975e-4 * interval_data["temperature"].values + 1.24965
    U_fitted = urc_inspect.scaler_U.unscale(c1 * X[:, 0] + c2 * X[:, 1] + c3 * X[:, 2] + c4 * X[:, 3] + c5) + U_ocv

    # Actual voltage in physical unit [V]
    U_actual = interval_data["voltage"].values
    
    # Compute residuals and identify outliers
    residuals = U_fitted - U_actual
    abs_res = np.abs(residuals)
    median_abs_res = np.median(abs_res)
    f_scale = eps_inspect * median_abs_res / 0.6745
    outlier_mask = abs_res > (eps_inspect * f_scale)
    
    # Plot inliers
    fig.add_trace(go.Scatter(
        x=U_actual[~outlier_mask],
        y=U_fitted[~outlier_mask],
        mode="markers",
        marker=dict(size=3, color="steelblue", opacity=0.5),
        name="Inliers" if i == 0 else None,
        legendgroup="inliers",
        showlegend=(i == 0),
    ), row=i+1, col=1)
    
    # Plot outliers
    if outlier_mask.sum() > 0:
        fig.add_trace(go.Scatter(
            x=U_actual[outlier_mask],
            y=U_fitted[outlier_mask],
            mode="markers",
            marker=dict(size=5, color="red", symbol="x"),
            name="Outliers" if i == 0 else None,
            legendgroup="outliers",
            showlegend=(i == 0),
        ), row=i+1, col=1)
    
    # Add diagonal line (perfect fit)
    u_range = [U_actual.min(), U_actual.max()]
    fig.add_trace(go.Scatter(
        x=u_range, y=u_range,
        mode="lines",
        line=dict(color="gray", dash="dash", width=1),
        name="Perfect Fit" if i == 0 else None,
        legendgroup="diagonal",
        showlegend=(i == 0),
    ), row=i+1, col=1)
    
    # Update axes
    fig.update_xaxes(title_text="U_actual [V]" if i == len(intervals_to_plot)-1 else "",
                     row=i+1, col=1)
    fig.update_yaxes(title_text="U_fitted [V]", row=i+1, col=1)

fig.update_layout(
    title=f"{extracted_name} — Outlier Visualization (ε={eps_inspect}, α={alpha_inspect})<br><sub>Actual vs Fitted Voltage</sub>",
    height=300 * len(intervals_to_plot),
    template="plotly_white",
)
fig.show()

## 5. Residual Plot with Outlier Highlighting

Visualize residuals vs fitted values to see outlier pattern.

In [13]:
# ── Residual plot ──
fig = make_subplots(
    rows=len(intervals_to_plot), cols=1,
    subplot_titles=[f"Interval {idx} (calh={rel.loc[idx, 'calh']:.0f}h, outlier={rel.loc[idx, 'outlier_pct']:.1f}%)" 
                    for idx in intervals_to_plot],
    vertical_spacing=0.08,
)

for i, idx in enumerate(intervals_to_plot):
    # Get interval metadata from reliable results
    row_meta = rel.loc[idx]
    if isinstance(row_meta, pd.DataFrame):
        row_meta = row_meta.iloc[0]

    day_since_install = row_meta.get("day_since_install", np.nan)

    if pd.isna(day_since_install):
        day_since_install = row_meta["calh"] / 24.0

    current_day = urc_inspect.installation_time + pd.Timedelta(days=float(day_since_install))
    interval_end = current_day + pd.Timedelta(days=urc_inspect.len_interval)

    # Slice original preprocessed data in the same time window as model fitting
    interval_data = urc_inspect.data.query("@current_day <= index <= @interval_end")

    if interval_data.empty:
        continue

    # Get coefficients
    c1, c2, c3, c4, c5 = [row_meta[f"c{j}"] for j in range(1, 6)]

    # Build features exactly like training
    X, _ = urc_inspect._prepare_matrices(interval_data)

    # Compute fitted voltage in physical unit [V]
    U_ocv = -8.2975e-4 * interval_data["temperature"].values + 1.24965
    U_fitted = urc_inspect.scaler_U.unscale(c1 * X[:, 0] + c2 * X[:, 1] + c3 * X[:, 2] + c4 * X[:, 3] + c5) + U_ocv

    # Actual voltage in physical unit [V]
    U_actual = interval_data["voltage"].values
    
    # Compute residuals and identify outliers
    residuals = U_fitted - U_actual
    abs_res = np.abs(residuals)
    median_abs_res = np.median(abs_res)
    f_scale = eps_inspect * median_abs_res / 0.6745
    outlier_mask = abs_res > (eps_inspect * f_scale)
    
    # Plot inlier residuals
    fig.add_trace(go.Scatter(
        x=U_fitted[~outlier_mask],
        y=residuals[~outlier_mask] * 1000,  # mV
        mode="markers",
        marker=dict(size=3, color="steelblue", opacity=0.5),
        name="Inliers" if i == 0 else None,
        legendgroup="inliers",
        showlegend=(i == 0),
    ), row=i+1, col=1)
    
    # Plot outlier residuals
    if outlier_mask.sum() > 0:
        fig.add_trace(go.Scatter(
            x=U_fitted[outlier_mask],
            y=residuals[outlier_mask] * 1000,
            mode="markers",
            marker=dict(size=5, color="red", symbol="x"),
            name="Outliers" if i == 0 else None,
            legendgroup="outliers",
            showlegend=(i == 0),
        ), row=i+1, col=1)
    
    # Add threshold lines
    threshold = eps_inspect * f_scale * 1000  # mV
    fig.add_hline(y=threshold, line_dash="dash", line_color="orange", opacity=0.5,
                  row=i+1, col=1)
    fig.add_hline(y=-threshold, line_dash="dash", line_color="orange", opacity=0.5,
                  row=i+1, col=1)
    fig.add_hline(y=0, line_color="gray", opacity=0.3, row=i+1, col=1)
    
    # Update axes
    fig.update_xaxes(title_text="U_fitted [V]" if i == len(intervals_to_plot)-1 else "",
                     row=i+1, col=1)
    fig.update_yaxes(title_text="Residual [mV]", row=i+1, col=1)

fig.update_layout(
    title=f"{extracted_name} — Residual Plot with Outliers (ε={eps_inspect}, α={alpha_inspect})",
    height=280 * len(intervals_to_plot),
    template="plotly_white",
)
fig.show()

## 6. Interactive Parameter Selector

Select different (epsilon, alpha) combinations to compare outlier patterns.

In [14]:
# ── Create interactive dropdown to switch between parameter combinations ──
from ipywidgets import interact, widgets

def plot_outliers_for_params(epsilon, alpha):
    """Plot outlier diagnostics for selected parameters."""
    urc = results[(epsilon, alpha)]
    rel = urc.fitting_results_reliable
    
    if rel.empty:
        print(f"No reliable results for ε={epsilon}, α={alpha}")
        return
    
    # Summary statistics
    print(f"\n{'='*60}")
    print(f"  ε={epsilon}, α={alpha}")
    print(f"{'='*60}")
    print(f"Reliable intervals: {len(rel)}")
    print(f"Median RMSE: {rel['RMSE'].median()*1000:.2f} mV")
    print(f"Median R²: {rel['R2'].median():.4f}")
    if "outlier_pct" in rel.columns:
        print(f"Mean outlier %: {rel['outlier_pct'].mean():.2f}%")
        print(f"Max outlier %: {rel['outlier_pct'].max():.2f}%")
    
    # Outlier time series
    if "outlier_pct" in rel.columns:
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=rel["calh"], y=rel["outlier_pct"],
            mode="markers+lines",
            marker=dict(size=4, color="#d62728"),
            line=dict(width=1),
        ))
        fig.update_layout(
            title=f"Outlier % Over Time (ε={epsilon}, α={alpha})",
            xaxis_title="Calendar hours [h]",
            yaxis_title="Outlier %",
            height=350,
            template="plotly_white",
        )
        fig.show()

# Create interactive widget
interact(
    plot_outliers_for_params,
    epsilon=widgets.Dropdown(options=epsilon_values, value=1.35, description="Epsilon:"),
    alpha=widgets.Dropdown(options=alpha_values, value=0.01, description="Alpha:"),
);

interactive(children=(Dropdown(description='Epsilon:', index=1, options=(1.0, 1.35, 2.0, 1000000), value=1.35)…

## 7. Export Results Summary

Save comparison table to CSV for further analysis.

In [15]:
# -- Export summary CSVs to master_arbeit_Di/plots/huber --
output_path = OUTPUT_DIR / f"sensitivity_analysis_{extracted_name}.csv"
df_comparison.to_csv(output_path, index=False)
print(f"✓ Saved to: {output_path}")

if "df_best_vs_baseline" in globals():
    path_best = OUTPUT_DIR / f"sensitivity_best_vs_baseline_{extracted_name}.csv"
    df_best_vs_baseline.to_csv(path_best, index=False)
    print(f"✓ Saved to: {path_best}")

if "compare_df_summary" in globals():
    path_compare = OUTPUT_DIR / f"sensitivity_compare_summary_{extracted_name}.csv"
    compare_df_summary.to_csv(path_compare, index=False)
    print(f"✓ Saved to: {path_compare}")

✓ Saved to: ..\plots\huber\sensitivity_analysis_G1M1_new.csv
✓ Saved to: ..\plots\huber\sensitivity_best_vs_baseline_G1M1_new.csv
✓ Saved to: ..\plots\huber\sensitivity_compare_summary_G1M1_new.csv


## 8. Unified Comparator Sensitivity Ranking (Cond + GT)

Use one unified metric table across **Baseline + all (epsilon, alpha)** combinations.

This section enables:
- `include_all_cond_metrics=True`
- `include_gt_metrics=True` (if GT files are available)

In [16]:
# -- Build unified models dict: baseline + all parameter combinations --
models_umc = {"Baseline": urc_baseline}
for (eps, alpha), m in results.items():
    models_umc[f"eps={eps}|alpha={alpha}"] = m

umc = UnifiedModelComparator(models_umc)

# Load GT per reference condition (aligned with unified compare notebook)
gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_path = Path(ref_cfg["gt_file"]).resolve()
    print(f"Looking for GT file ({ref_name}): {gt_path}")

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(f"Cannot identify voltage column in {gt_path}. Columns: {gt_data.columns.tolist()}")

            gt_series = gt_series.dropna()
            umc.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(f"  ✓ Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [column: {selected_col}]")
        except Exception as e:
            print(f"  ✗ Failed to load GT for {ref_name}: {e}")
    else:
        print(f"  ✗ GT file NOT found for {ref_name}: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
if SHOW_GT_METRICS and not has_gt:
    raise ValueError("GT metrics is enabled, but no GT files were loaded. Check REF_CONFIGS gt_file paths.")

print(f"GT loaded: {gt_loaded_count}/{len(REF_CONFIGS)} refs; GT enabled: {has_gt}")

# Aggregate metrics across Iref
rows = []
for i_ref in common_config["Iref"]:
    df_i = umc.compare_all(
        i_target=i_ref,
        outlier_threshold_method="2rmse",
        include_gt_metrics=has_gt,
        include_all_cond_metrics=True,
    ).copy()
    if df_i.empty:
        continue
    df_i["Iref"] = i_ref
    rows.append(df_i)

if not rows:
    raise ValueError("No comparison rows produced by UnifiedModelComparator.")

df_umc_all = pd.concat(rows, ignore_index=True)
path_umc = OUTPUT_DIR / f"sensitivity_umc_all_{extracted_name}.csv"
df_umc_all.to_csv(path_umc, index=False)
print(f"✓ Saved to: {path_umc}")
display(df_umc_all.head())

Looking for GT file (Low): C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv
  ✓ Loaded GT for Low (Iref=0.3): 1762 points [column: gt_uref_regression]
Looking for GT file (Medium): C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv
  ✓ Loaded GT for Medium (Iref=1.0): 1762 points [column: gt_uref_regression]
Looking for GT file (High): C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv
  ✓ Loaded GT for High (Iref=1.48): 1384 points [column: gt_uref_regression]
GT loaded: 3/3 refs; GT enabled: True
✓ Saved to: ..\plots\huber\sensitivity_umc_all_G1M1

,Model Name,Target Current (A/cm2),Fitting Time (s),Data Points (n),Degradation Rate (uV/h),RMSE (mV),Slope Sigma (uV/h),Mono (Rank) [0-1],Outlier Count,Outlier (%),Max Residual (mV),Mean SE (mV),Cond Median log10,Cond P95 log10,Cond Count (Used Scope),Cond Scope,GT RMSE (mV),GT MAE (mV),GT Valid Intervals,Iref
0,Baseline,0.3,12.777,1098,1.252113,18.885,0.000,0.925,10,0.91,390.861,4.105,4.745,13.287,1286,all_fitted,19.114,5.783,1098,0.3
1,eps=1.0|alpha=0.0,0.3,22.688,1081,1.186625,17.781,0.022,0.933,10,0.93,378.709,2.514,5.007,13.282,1286,all_fitted,18.022,5.484,1081,0.3
2,eps=1.0|alpha=0.01,0.3,23.170,1256,1.262646,51.198,0.029,0.685,61,4.86,519.486,2.566,4.717,5.787,1286,all_fitted,49.777,15.287,1256,0.3
3,eps=1.0|alpha=0.1,0.3,24.128,1277,0.914401,59.357,0.063,0.604,84,6.58,500.117,2.448,4.077,4.878,1286,all_fitted,59.894,19.217,1277,0.3
4,eps=1.0|alpha=1,0.3,27.862,1274,0.006842,50.779,0.114,0.599,84,6.59,455.436,2.558,3.131,3.962,1286,all_fitted,54.599,17.716,1274,0.3


In [17]:
# ── Parse parameter labels and compute robust composite ranking ──
def parse_eps_alpha(model_name: str):
    if model_name == "Baseline":
        return np.nan, np.nan
    m = re.match(r"eps=([0-9eE+\-.]+)\|alpha=([0-9eE+\-.]+)", str(model_name))
    if m:
        return float(m.group(1)), float(m.group(2))
    return np.nan, np.nan

df_rank = df_umc_all.copy()
eps_alpha = df_rank["Model Name"].apply(parse_eps_alpha)
df_rank["epsilon"] = eps_alpha.apply(lambda x: x[0])
df_rank["alpha"] = eps_alpha.apply(lambda x: x[1])

# Average over Iref for each model
agg_cols = [
    "RMSE (mV)", "Slope Sigma (uV/h)", "Outlier (%)",
    "Cond Median log10", "Mean SE (mV)"
]
if has_gt and "GT RMSE (mV)" in df_rank.columns:
    agg_cols.append("GT RMSE (mV)")

df_model = (
    df_rank.groupby(["Model Name", "epsilon", "alpha"], dropna=False)[agg_cols]
    .mean(numeric_only=True)
    .reset_index()
)

# Composite score: lower is better
score_terms = []
for col in ["RMSE (mV)", "Slope Sigma (uV/h)", "Outlier (%)", "Cond Median log10", "Mean SE (mV)"] + (["GT RMSE (mV)"] if has_gt and "GT RMSE (mV)" in df_model.columns else []):
    s = df_model[col].astype(float)
    z = (s - s.mean()) / (s.std(ddof=0) + 1e-12)
    score_terms.append(z)

df_model["composite_score"] = np.sum(np.column_stack(score_terms), axis=1)
df_model = df_model.sort_values("composite_score", ascending=True).reset_index(drop=True)

print("Top models by composite score (lower is better):")
display(df_model.head(15))

# Best parameter set (excluding baseline)
df_non_base = df_model[df_model["Model Name"] != "Baseline"].copy()
if not df_non_base.empty:
    best = df_non_base.iloc[0]
    print("\nBest parameter candidate:")
    print(f"  epsilon={best['epsilon']}, alpha={best['alpha']}")
    print(f"  composite_score={best['composite_score']:.3f}")

Top models by composite score (lower is better):


,Model Name,epsilon,alpha,RMSE (mV),Slope Sigma (uV/h),Outlier (%),Cond Median log10,Mean SE (mV),GT RMSE (mV),composite_score
0,Baseline,NaN,NaN,30.967000,0.000000,3.520000,4.745,4.531667,34.730333,-5.402959
1,eps=1.0|alpha=0.0,1.00,0.00,31.953333,0.052667,3.180000,5.007,3.062000,34.702000,-4.826986
2,eps=1.35|alpha=0.0,1.35,0.00,31.957667,0.051667,3.380000,4.955,3.060333,34.855667,-4.720381
3,eps=2.0|alpha=0.0,2.00,0.00,31.866333,0.052333,3.466667,4.895,3.078333,34.797333,-4.690450
4,eps=1000000|alpha=0.0,1000000.00,0.00,32.226000,0.053333,3.733333,4.745,3.101333,34.730333,-4.568033
5,eps=1.35|alpha=100,1.35,100.00,101.216000,0.210000,3.306667,1.168,3.346333,104.690667,-0.735843
6,eps=2.0|alpha=100,2.00,100.00,100.849333,0.209000,3.306667,1.239,3.482667,104.056000,-0.713467
7,eps=1.35|alpha=0.01,1.35,0.01,94.070333,0.069333,3.883333,4.718,3.099333,98.524333,-0.199879
8,eps=1.0|alpha=0.01,1.00,0.01,93.772667,0.069667,3.980000,4.717,3.064000,98.154333,-0.146079
9,eps=2.0|alpha=0.01,2.00,0.01,96.235667,0.069667,3.936667,4.698,3.160667,100.925333,-0.014397



Best parameter candidate:
  epsilon=1.0, alpha=0.0
  composite_score=-4.827


In [18]:
# ── Heatmaps by Iref: RMSE / Cond / GT RMSE (if available) ──
for i_ref in common_config["Iref"]:
    dfi = df_rank[df_rank["Iref"] == i_ref].copy()
    dfi = dfi.dropna(subset=["epsilon", "alpha"])
    if dfi.empty:
        continue

    pivot_rmse = dfi.pivot_table(index="epsilon", columns="alpha", values="RMSE (mV)", aggfunc="mean")
    fig = go.Figure(data=go.Heatmap(
        z=pivot_rmse.values,
        x=[str(c) for c in pivot_rmse.columns],
        y=[str(r) for r in pivot_rmse.index],
        colorscale="Viridis",
        colorbar=dict(title="RMSE (mV)"),
    ))
    fig.update_layout(
        title=f"Unified Sensitivity @ Iref={i_ref} — RMSE",
        xaxis_title="alpha",
        yaxis_title="epsilon",
        template="plotly_white",
        height=420,
    )
    fig.show()

    if "Cond Median log10" in dfi.columns:
        pivot_cond = dfi.pivot_table(index="epsilon", columns="alpha", values="Cond Median log10", aggfunc="mean")
        fig = go.Figure(data=go.Heatmap(
            z=pivot_cond.values,
            x=[str(c) for c in pivot_cond.columns],
            y=[str(r) for r in pivot_cond.index],
            colorscale="Cividis",
            colorbar=dict(title="Cond Median log10"),
        ))
        fig.update_layout(
            title=f"Unified Sensitivity @ Iref={i_ref} — Cond Median log10",
            xaxis_title="alpha",
            yaxis_title="epsilon",
            template="plotly_white",
            height=420,
        )
        fig.show()

    if has_gt and "GT RMSE (mV)" in dfi.columns:
        pivot_gt = dfi.pivot_table(index="epsilon", columns="alpha", values="GT RMSE (mV)", aggfunc="mean")
        fig = go.Figure(data=go.Heatmap(
            z=pivot_gt.values,
            x=[str(c) for c in pivot_gt.columns],
            y=[str(r) for r in pivot_gt.index],
            colorscale="Plasma",
            colorbar=dict(title="GT RMSE (mV)"),
        ))
        fig.update_layout(
            title=f"Unified Sensitivity @ Iref={i_ref} — GT RMSE",
            xaxis_title="alpha",
            yaxis_title="epsilon",
            template="plotly_white",
            height=420,
        )
        fig.show()